# Automatización Fase 4 (Consolidación) + Fase 5 (Freeze)

Continúa después de `Automatizacion_Fase3_Asignacion_Instructor_Vuelos.ipynb` (que ya escribió
instructor + "LCK A320F" en los bloques). Este notebook reconstruye los mismos bloques/asignaciones
(mismo BigQuery + mismo matching contra la Matriz, sin cambios) y agrega escrituras nuevas,
**todas en modo vista previa primero**:

1. **Reemplazar el slot `"LCK A320F"` de la Matriz** por el detalle de vuelos, formato confirmado
   por Fernando (secciones 1-6).
2. **Armar el CUADRO FINAL en Archivo 10** (Fecha, DíaSEM, Vuelo, Ruta, N° Cupos, INS — Grupo
   vacío), la tabla auxiliar (INS/Cantidad/Grupos con fórmula) y la lista de instructores no
   considerados (secciones 7-12).
3. **Por tripulante**, cruzar "INS FINAL" (Y) contra los 4 grupos para calcular "INS F a
   considerar" (Z) y "Grupo" (AA) (secciones 13-16).
4. **Fase 5 (Freeze)**: armar el roster de tripulantes + bloque de vuelos + tabla alterna del
   CUADRO FINAL, para la hoja de ejemplo del Freeze (secciones 17-21).

**Lo que sigue SIN automatizar (confirmado, no se inventa):** los 4 grupos de instructores
(`AD2`/`AE2`/`AF2`/`AG2`, los escribe Fernando a mano cada mes), la columna Y "INS FINAL" (ya
resuelta por otro proceso), y la conciliación cupos-vs-demanda / priorización por vencimiento /
envío final del Freeze (pasos 23-25 del documento) — ver la sección final "Estado y pendientes"
para el detalle completo.


## 1. Instalar dependencias y autenticar

In [ ]:
!pip install -q gspread


In [ ]:
import sys
import re
import unicodedata
import collections
from collections import defaultdict
import pandas as pd
from google.colab import auth
from google.cloud import bigquery
import gspread
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
# Mismo motivo que en la Fase 3: la cuenta de Fernando no tiene "bigquery.jobs.create"
# en operations-data-prod, pero sí en datadem-home (proyecto de facturación).
client = bigquery.Client(project="datadem-home")
print("Autenticado: BigQuery + Google Sheets con tu cuenta.")


## 2. Query a BigQuery + armado de candidatos/bloques (igual que Fase 3, sin cambios)

In [ ]:
MES_OBJETIVO = 10
ANIO_OBJETIVO = 2026

WD_ES = {
    "Monday": "lunes", "Tuesday": "martes", "Wednesday": "miércoles",
    "Thursday": "jueves", "Friday": "viernes", "Saturday": "sábado",
    "Sunday": "domingo",
}

RUTAS_VALIDAS_ARR = {"AQP", "CIX", "CJA", "CUZ", "PEM", "PIU", "TBP", "TPP", "TCQ"}
EXCLUSIONES_MES = {"AQP"}  # confirmado con Fernando: sigue vigente en octubre-2026

HORA_MIN_SALIDA = pd.Timedelta(hours=8, minutes=30)
HBT_MIN = pd.Timedelta(hours=1)
CONEXION_MIN = pd.Timedelta(minutes=50)
CONEXION_MAX = pd.Timedelta(hours=1, minutes=30)
PSV_MAX = pd.Timedelta(hours=11)

QUERY_NB = """
SELECT
  pairing_id                       AS trip,
  flight_start_date_local_time     AS inicio_vuelo_lt,
  duty_calendar_day_number         AS dia_duty,
  flight_number                    AS vuelo,
  departure_airport_code           AS dep,
  arrival_airport_code             AS arr,
  flight_departure_time_crew_base  AS std_hb,
  flight_arrival_hour_block_time   AS sta_hb,
  flight_block_time                AS hbt,
  subfleet_code                    AS sub_fleet

FROM `operations-data-prod.carmen_gold.crew_pairing_carmen_system`

WHERE
  flight_start_date_local_time BETWEEN DATE '2025-10-01' AND DATE '2026-10-31'
  AND subsidiary_code IN ('LP')
  AND load_type_code = 'FP'
  AND crew_range_type_code = 'SAB'
  AND subfleet_code IN ('319', '320')

QUALIFY
  CASE
    WHEN load_type_code = 'FP' AND
         DATE(ingestion_datetime) = MAX(CASE WHEN load_type_code = 'FP' THEN DATE(ingestion_datetime) END)
           OVER (PARTITION BY subsidiary_code, fleet_type_code, crew_range_type_code,
                              reference_month_number, reference_year)
      THEN 0
    WHEN load_type_code = 'ES' AND
         MAX(CASE WHEN load_type_code = 'FP' THEN 0 ELSE 0 END)
           OVER (PARTITION BY subsidiary_code, fleet_type_code, crew_range_type_code,
                              reference_month_number, reference_year) = -1 AND
         DATE(ingestion_datetime) = MAX(CASE WHEN load_type_code = 'ES' THEN DATE(ingestion_datetime) END)
           OVER (PARTITION BY subsidiary_code, fleet_type_code, crew_range_type_code,
                              reference_month_number, reference_year)
      THEN 0
    ELSE -1
  END = 0

ORDER BY pairing_id ASC
"""

df_raw = client.query(QUERY_NB).to_dataframe(create_bqstorage_client=False)
print(f"Filas descargadas: {len(df_raw)}")


In [ ]:
def parse_hora(s):
    if pd.isna(s):
        return pd.NaT
    h, m, sec = str(s).split(":")
    sec = sec.split(".")[0]
    return pd.Timedelta(hours=int(h), minutes=int(m), seconds=int(sec))


def cargar_bq_a_df(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()
    df["trip"] = df["trip"].astype(str)
    df["dia_duty"] = df["dia_duty"].astype(int)
    df["fecha_dt"] = pd.to_datetime(df["inicio_vuelo_lt"])
    df["std_td"] = df["std_hb"].apply(parse_hora)
    df["sta_td"] = df["sta_hb"].apply(parse_hora)
    df["hbt_td"] = df["hbt"].apply(parse_hora)
    df["std_dt"] = df["fecha_dt"] + df["std_td"]
    df["sta_dt"] = df["fecha_dt"] + df["sta_td"]
    df.loc[df["sta_dt"] < df["std_dt"], "sta_dt"] += pd.Timedelta(days=1)
    df["dia_semana"] = df["fecha_dt"].dt.day_name().map(WD_ES)
    return df


def separar_instancias_trip(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["trip", "fecha_dt", "std_dt"]).reset_index(drop=True)
    instancia = []
    trip_actual = None
    max_dia_duty = -1
    idx_instancia = 0
    for _, row in df.iterrows():
        if row["trip"] != trip_actual:
            trip_actual = row["trip"]
            idx_instancia = 0
            max_dia_duty = row["dia_duty"]
        elif row["dia_duty"] < max_dia_duty:
            idx_instancia += 1
            max_dia_duty = row["dia_duty"]
        else:
            max_dia_duty = max(max_dia_duty, row["dia_duty"])
        trip_val = row["trip"]
        instancia.append(f"{trip_val}_{idx_instancia}")

    df["trip_original"] = df["trip"]
    df["trip"] = instancia
    return df


def filtrar_mes_y_ruta(df: pd.DataFrame, mes: int, anio: int) -> pd.DataFrame:
    rutas_validas = RUTAS_VALIDAS_ARR - {c.upper() for c in EXCLUSIONES_MES}
    en_mes = (df["fecha_dt"].dt.month == mes) & (df["fecha_dt"].dt.year == anio)
    sale_de_lim = df["dep"] == "LIM"
    llega_a_lim = df["arr"] == "LIM"
    ruta_nacional_ok = df["arr"].isin(rutas_validas) | (llega_a_lim)
    return df[en_mes & (sale_de_lim | llega_a_lim) & ruta_nacional_ok].copy()


def armar_primeras_mitades(df: pd.DataFrame, dia_duty_min_real=None):
    validos_rows = []
    excluidos_rows = []

    for trip, grupo in df.groupby("trip"):
        dia_min = grupo["dia_duty"].min()
        if dia_duty_min_real is not None and trip in dia_duty_min_real.index and dia_min != dia_duty_min_real.loc[trip]:
            excluidos_rows.append({"trip": trip, "motivo": "día 1 real fuera de mes/ruta"})
            continue
        dia1 = grupo[grupo["dia_duty"] == dia_min].sort_values("std_dt")

        if len(dia1) < 2:
            excluidos_rows.append({"trip": trip, "motivo": "día 1 sin vuelta el mismo día"})
            continue
        if len(dia1) in (6, 8, 10):
            excluidos_rows.append({"trip": trip, "motivo": f"día 1 tiene {len(dia1)} tramos"})
            continue

        ida, vuelta = dia1.iloc[0], dia1.iloc[1]
        if ida["dep"] != "LIM":
            excluidos_rows.append({"trip": trip, "motivo": "el primer tramo no sale de LIM"})
            continue
        if vuelta["arr"] != "LIM":
            excluidos_rows.append({"trip": trip, "motivo": "el segundo tramo no vuelve a LIM"})
            continue
        if vuelta["dep"] != ida["arr"]:
            excluidos_rows.append({"trip": trip, "motivo": "ruta triangular"})
            continue

        motivos = []
        if (ida["std_dt"] - ida["fecha_dt"]) <= HORA_MIN_SALIDA:
            motivos.append("sale antes/igual a 08:30")
        if ida["hbt_td"] <= HBT_MIN:
            motivos.append("HBT ida <= 1h")
        if vuelta["hbt_td"] <= HBT_MIN:
            motivos.append("HBT vuelta <= 1h")
        conexion = vuelta["std_dt"] - ida["sta_dt"]
        if conexion <= pd.Timedelta(0):
            motivos.append("conexión interna negativa/cero")
        psv = vuelta["sta_dt"] - ida["std_dt"]
        if psv > PSV_MAX:
            motivos.append(f"PSV {psv} > 11h")

        if motivos:
            excluidos_rows.append({"trip": trip, "motivo": "; ".join(motivos)})
            continue

        validos_rows.append({
            "Fecha": ida["fecha_dt"].strftime("%d/%m/%Y"),
            "DíaSem": ida["dia_semana"],
            "Pairing ID": trip,
            "Vuelo Ida": ida["vuelo"], "Dep": ida["dep"], "Arr": ida["arr"],
            "STD Ida": ida["std_hb"], "STA Ida": ida["sta_hb"], "HBT Ida": ida["hbt"],
            "Vuelo Vuelta": vuelta["vuelo"], "Dep Vta": vuelta["dep"], "Arr Vta": vuelta["arr"],
            "STD Vuelta": vuelta["std_hb"], "STA Vuelta": vuelta["sta_hb"], "HBT Vuelta": vuelta["hbt"],
            "Conexión": str(conexion), "PSV Total": str(psv),
            "Sub Flota": ida["sub_fleet"],
            "_orden": ida["std_dt"],
        })

    cols_finales = ["Fecha", "DíaSem", "Pairing ID", "Vuelo Ida", "Dep", "Arr", "STD Ida",
                     "STA Ida", "HBT Ida", "Vuelo Vuelta", "Dep Vta", "Arr Vta", "STD Vuelta",
                     "STA Vuelta", "HBT Vuelta", "Conexión", "PSV Total", "Sub Flota"]

    if not validos_rows:
        return pd.DataFrame(columns=cols_finales), pd.DataFrame(excluidos_rows)

    validos = pd.DataFrame(validos_rows).sort_values("_orden")
    validos = validos[cols_finales].reset_index(drop=True)
    return validos, pd.DataFrame(excluidos_rows)


def _fecha_dt(s):
    d, m, a = s.split("/")
    return pd.Timestamp(year=int(a), month=int(m), day=int(d))


def _hora_td(s):
    h, m, sec = str(s).split(":")
    sec = sec.split(".")[0]
    return pd.Timedelta(hours=int(h), minutes=int(m), seconds=int(sec))


def parear_candidatos(validos: pd.DataFrame):
    df = validos.copy()
    df["_ida_dt"] = df.apply(lambda r: _fecha_dt(r["Fecha"]) + _hora_td(r["STD Ida"]), axis=1)
    df["_vta_dt"] = df.apply(lambda r: _fecha_dt(r["Fecha"]) + _hora_td(r["STA Vuelta"]), axis=1)
    df = df.sort_values("_ida_dt").reset_index(drop=True)

    usados = set()
    bloques = []
    for i, row in df.iterrows():
        if row["Pairing ID"] in usados:
            continue
        ventana_ini = row["_vta_dt"] + CONEXION_MIN
        ventana_fin = row["_vta_dt"] + CONEXION_MAX
        mismo_dia = df[
            (~df["Pairing ID"].isin(usados)) &
            (df["Pairing ID"] != row["Pairing ID"]) &
            (df["Fecha"] == row["Fecha"]) &
            (df["_ida_dt"] > ventana_ini) & (df["_ida_dt"] < ventana_fin) &
            (df["_vta_dt"] - row["_ida_dt"] <= PSV_MAX)
        ].sort_values("_ida_dt")

        usados.add(row["Pairing ID"])
        if len(mismo_dia) > 0:
            segunda = mismo_dia.iloc[0]
            usados.add(segunda["Pairing ID"])
            bloques.append((row, segunda))
        else:
            bloques.append((row, None))
    return bloques


df = cargar_bq_a_df(df_raw)
df = separar_instancias_trip(df)
dia_duty_min_real = df.groupby("trip")["dia_duty"].min()
df_filtrado = filtrar_mes_y_ruta(df, MES_OBJETIVO, ANIO_OBJETIVO)
validos, excluidos = armar_primeras_mitades(df_filtrado, dia_duty_min_real)
bloques = parear_candidatos(validos)

n_parejas = sum(1 for _, b in bloques if b is not None)
n_solos = sum(1 for _, b in bloques if b is None)
print(f"Candidatos válidos: {len(validos)}")
print(f"Bloques armados: {len(bloques)} ({n_parejas} completos, {n_solos} solos)")


## 3. Leer la Matriz real (con la fila/columna EXACTA de cada slot "LCK A320F")

Igual que en la Fase 3, pero ahora también se guarda `fila_hoja`/`col_hoja` de cada slot -> se
necesita para poder escribir de vuelta en la celda correcta.

In [ ]:
URL_MATRIZ = "https://docs.google.com/spreadsheets/d/19WmwaoLDZnArNztu_dJNwi7bjrGq-_cx_gw0aN96zfk/edit?gid=580414308"

sh_matriz = gc.open_by_url(URL_MATRIZ)
ws_matriz = sh_matriz.get_worksheet_by_id(580414308)

FILA_ENCABEZADO_FECHAS = 2
FILA_PRIMER_INSTRUCTOR = 3
COL_PRIMERA_FECHA = 3

valores_m = ws_matriz.get_all_values()

fila_fechas = valores_m[FILA_ENCABEZADO_FECHAS - 1]
fechas_matriz = []
for celda in fila_fechas[COL_PRIMERA_FECHA - 1:]:
    fechas_matriz.append(celda.strip() if celda.strip() else None)

reservas = []  # (bp, nombre_matriz, fecha_str, fila_hoja, col_hoja)
for i, fila in enumerate(valores_m[FILA_PRIMER_INSTRUCTOR - 1:]):
    fila_hoja = FILA_PRIMER_INSTRUCTOR + i
    if not fila or not fila[0].strip():
        continue
    bp = fila[0].strip().lstrip("'")
    nombre = fila[1].strip() if len(fila) > 1 else ""
    for j, celda in enumerate(fila[COL_PRIMERA_FECHA - 1:]):
        col_hoja = COL_PRIMERA_FECHA + j
        if celda.strip().upper() == "LCK A320F":
            fecha_str = fechas_matriz[j] if j < len(fechas_matriz) else None
            if fecha_str:
                reservas.append((bp, nombre, fecha_str, fila_hoja, col_hoja))

print(f"Slots 'LCK A320F' encontrados en la Matriz: {len(reservas)}")


## 4. Emparejar reservas con bloques (misma lógica de la Fase 3)

Mismas reglas confirmadas: solo bloque completo en la fecha exacta; nombre normalizado
(tildes/mayúsculas); si algo no calza, se reporta, no se inventa.

In [ ]:
INSTRUCTORES_DATA = [
    ("Christian Rondon", "1271571", " RONDON BARRUTIA CHRISTIAN ERIC "),
    ("Erika Davila", "967092", "DAVILA BELLO MARIA ERIKA"),
    ("Sebastian Correa", "2396710", " CORREA GARCIA JUAN SEBASTIAN "),
    ("Fiorella Ruiz", "2713993", "RUIZ RIOJA FIORELLA DEL PILAR"),
    ("Jazmin Guerra", "29530", "GUERRA SUAREZ JAZMIN"),
    ("Jennifert Acurio", "3779550", "ACURIO DARGENT JENNIFERT MILAGROS"),
    ("Karen Santa Cruz", "2843319", "SANTA CRUZ HUAMAN KAREN"),
    ("Luis Bacigalupo", "2963161", "BACIGALUPO FLORES LUIS ENRIQUE"),
    ("Claudia Flores", "3217561", " FLORES FUENTES DAVILA CLAUDIA ALEXANDRA "),
    ("Karla Moz", "71348", "MOZ MONTES KARLA LISSETTE"),
    ("Elizabeth Torres", "2369641", "TORRES POLO ELIZABETH DEL PILAR"),
    ("Patricia Najar", "2369624", "NAJAR CRUZ PATRICIA DEL PILAR"),
    ("Javier Zapata", "3134911", "ZAPATA GARAYAR JAVIER RICARDO SALVADOR"),
    ("Jefferson Mendez", "3750335", " MENDEZ RUCOBA JEFFERSON "),
    ("Gabriela Ungaro", "3852423", "UNGARO GUTIERREZ GABRIELA"),
    ("Mariella Carrasco", "2604360", "CARRASCO BENAVIDES ROSA MARIELLA"),
    ("Cesar Campos", "2823133", " CAMPOS CONCHE CESAR AUGUSTO "),
    ("Kevin Segovia", "3189967", "SEGOVIA TAPIA RAY KEVIN"),
    ("Milagros Salas", "2415373", "SALAS COSIO MILAGROS PATRICIA"),
    ("Judith Fernandez", "2440915", "FERNANDEZ GARCIA JUDITH JULIET"),
    ("Rafael Nieto", "3796947", " NIETO SAENZ RAFAEL ANTONIO "),
    ("Gianfranco Celiz", "3841387", " CELIZ ROSSI GIANFRANCO PAOLO "),
]


def normalizar_nombre(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return s.strip().lower()


def normalizar_fecha(fecha_str):
    d, m, a = fecha_str.strip().split("/")
    return (int(d), int(m), int(a))


def emparejar_matriz_con_bloques(reservas, bloques):
    mapa_nombre_canonico = {normalizar_nombre(n): n for n, _bp, _legal in INSTRUCTORES_DATA}

    bloques_por_fecha = defaultdict(list)
    for idx, (candA, _candB) in enumerate(bloques):
        bloques_por_fecha[normalizar_fecha(candA["Fecha"])].append(idx)

    usados_bloques = set()
    asignaciones_por_bloque = {}
    celda_matriz_por_bloque = {}
    sin_bloque = []
    sin_match_nombre = []

    for bp, nombre_matriz, fecha_str, fila_hoja, col_hoja in reservas:
        nombre_canonico = mapa_nombre_canonico.get(normalizar_nombre(nombre_matriz))
        if nombre_canonico is None:
            sin_match_nombre.append((bp, nombre_matriz, fecha_str))
            continue

        fecha_norm = normalizar_fecha(fecha_str)
        candidatos_idx = [i for i in bloques_por_fecha.get(fecha_norm, []) if i not in usados_bloques]
        candidatos_completos = [i for i in candidatos_idx if bloques[i][1] is not None]

        elegido = candidatos_completos[0] if candidatos_completos else None
        if elegido is None:
            sin_bloque.append((bp, nombre_canonico, fecha_str))
            continue

        usados_bloques.add(elegido)
        asignaciones_por_bloque[elegido] = (nombre_canonico, "LCK A320F")
        celda_matriz_por_bloque[elegido] = (fila_hoja, col_hoja)

    return asignaciones_por_bloque, celda_matriz_por_bloque, sin_bloque, sin_match_nombre


asignaciones_por_bloque, celda_matriz_por_bloque, sin_bloque, sin_match_nombre = emparejar_matriz_con_bloques(reservas, bloques)

print(f"Reservas de la Matriz: {len(reservas)}")
print(f"Asignadas a un bloque: {len(asignaciones_por_bloque)}")
print(f"Sin bloque completo esa fecha (revisar a mano): {len(sin_bloque)}")
for bp, nombre, fecha in sin_bloque:
    print(f"  - {nombre} (BP {bp}) reservado el {fecha}")
print(f"Sin match de nombre (revisar a mano): {len(sin_match_nombre)}")
for bp, nombre, fecha in sin_match_nombre:
    print(f"  - '{nombre}' (BP {bp}, {fecha})")


## 5. Vista previa: detalle de vuelos para reemplazar cada slot "LCK A320F" en la Matriz

Formato confirmado por Fernando: dentro de la MISMA celda, las 2 mitades del bloque una
debajo de otra (actividad / ruta / vuelo ida / vuelo vuelta), separadas por salto de línea.

In [ ]:
def construir_texto_matriz(candA, candB, actividad="LCK A320F"):
    partes = []
    for cand in [candA] + ([candB] if candB is not None else []):
        ruta = f"LIM-{cand['Arr']}-LIM"
        leg_ida = f"LA {cand['Vuelo Ida']} ({str(cand['STD Ida'])[:5]}-{str(cand['STA Ida'])[:5]} hrs)"
        leg_vta = f"LA {cand['Vuelo Vuelta']} ({str(cand['STD Vuelta'])[:5]}-{str(cand['STA Vuelta'])[:5]} hrs)"
        partes.extend([actividad, ruta, leg_ida, leg_vta])
    return "\n".join(partes)


actualizaciones_matriz = []  # (fila_hoja, col_hoja, texto)
for idx_bloque, (nombre, actividad) in asignaciones_por_bloque.items():
    candA, candB = bloques[idx_bloque]
    fila_hoja, col_hoja = celda_matriz_por_bloque[idx_bloque]
    texto = construir_texto_matriz(candA, candB, actividad)
    actualizaciones_matriz.append((fila_hoja, col_hoja, texto))

print(f"Celdas de la Matriz a actualizar: {len(actualizaciones_matriz)}\n")
for fila_hoja, col_hoja, texto in actualizaciones_matriz:
    print(f"--- Matriz fila {fila_hoja}, columna {col_hoja} ---")
    print(texto)
    print()


## 6. Escribir en la Matriz real — DESACTIVADO por defecto

In [ ]:
# --- DESCOMENTAR SOLO DESPUES DE REVISAR LA VISTA PREVIA DE LA CELDA ANTERIOR ---

# celdas_gs = [gspread.Cell(row=f, col=c, value=t) for f, c, t in actualizaciones_matriz]
# ws_matriz.update_cells(celdas_gs, value_input_option="USER_ENTERED")
# print(f"Actualizadas {len(celdas_gs)} celdas en la Matriz.")


## 7. Vista previa: CUADRO FINAL para Archivo 10 (pasos 17-19)

Solo se incluyen asignaciones con actividad `"LCK A320F"` o `"LCK A320F + Habilitación A320F"`.
**Cambio confirmado por Fernando:** la columna "Grupo" del CUADRO FINAL (AH) queda **vacía** —
los grupos ya NO los fija este notebook. Fernando los va a escribir a mano en 4 celdas (listas de
nombres separadas por coma): `AD2`=Grupo 1, `AF2`=Grupo 2, `AG2`=Grupo 3, `AH2`=Grupo 4 (opcional).
Más abajo (sección 7b) se arma una tabla auxiliar con una fórmula que lee esas 4 celdas.


In [ ]:
ACTIVIDADES_CUADRO_FINAL = {"LCK A320F", "LCK A320F + Habilitación A320F"}

filas_cuadro_final = []  # dicts: Fecha, DiaSem, Vuelo, Ruta, Cupos, Grupo (vacío), INS

for idx_bloque, (nombre, actividad) in asignaciones_por_bloque.items():
    if actividad not in ACTIVIDADES_CUADRO_FINAL:
        continue
    candA, candB = bloques[idx_bloque]

    for cand in [candA] + ([candB] if candB is not None else []):
        cupos = 4 if str(cand["Sub Flota"]) == "320" else 3
        filas_cuadro_final.append({
            "Fecha": cand["Fecha"],
            "DiaSem": cand["DíaSem"],
            "Vuelo": f"{cand['Vuelo Ida']}/{cand['Vuelo Vuelta']}",
            "Ruta": f"LIM-{cand['Arr']}-LIM",
            "Cupos": cupos,
            "Grupo": "",  # confirmado: se deja vacío, Fernando arma los grupos a mano
            "INS": nombre,
        })

print(f"Filas para el CUADRO FINAL: {len(filas_cuadro_final)}")
print("\nPrimeras filas:")
for f in filas_cuadro_final[:6]:
    print(f)


## 8. Escribir el CUADRO FINAL en Archivo 10 — DESACTIVADO por defecto

In [ ]:
URL_ARCHIVO_10_LCK320F = "https://docs.google.com/spreadsheets/d/1NZN565fOJUtoETQvvPzdHpRvyHp4stY2hsrqtjNjSEU/edit?gid=1262242775"
sh_archivo10 = gc.open_by_url(URL_ARCHIVO_10_LCK320F)
ws_archivo10_lck320f = sh_archivo10.get_worksheet_by_id(1262242775)

FILA_INICIO_CUADRO_FINAL = 10  # confirmado: encabezados en fila 9, datos desde fila 10
COL_INICIO_CUADRO_FINAL = "AC"

valores_a_escribir = [
    [f["Fecha"], f["DiaSem"], f["Vuelo"], f["Ruta"], f["Cupos"], f["Grupo"], f["INS"]]
    for f in filas_cuadro_final
]
rango_cuadro_final = (
    f"{COL_INICIO_CUADRO_FINAL}{FILA_INICIO_CUADRO_FINAL}:"
    f"AI{FILA_INICIO_CUADRO_FINAL + len(valores_a_escribir) - 1}"
)
print(f"Se escribirían {len(valores_a_escribir)} filas en Archivo 10, rango {rango_cuadro_final}")

# --- DESCOMENTAR SOLO DESPUES DE REVISAR LA VISTA PREVIA ---
# if valores_a_escribir:
#     ws_archivo10_lck320f.update(rango_cuadro_final, valores_a_escribir)
#     print(f"Escritas {len(valores_a_escribir)} filas en Archivo 10.")


## 9. Tabla auxiliar (INS / Cantidad / Grupos) + lista de instructores no considerados

Esto es lo que le da sentido a dejar "Grupo" vacío en el CUADRO FINAL: en vez de que este
notebook decida los grupos, Fernando los escribe a mano (listas separadas por coma) en:
`AD2`=Grupo 1, `AE2`=Grupo 2, `AF2`=Grupo 3, `AG2`=Grupo 4 (confirmado por Fernando, con la
fórmula real que ya puso en la hoja: `SI(ESNUMERO(HALLAR(...)),...)`).

- **Tabla auxiliar** (`AK14:AM...`, mismo lugar que el `AK15:AM27` del manual original): una fila
  por instructor que SÍ tiene vuelos este mes, con la cantidad de vuelos y una **fórmula** en
  "Grupos" que busca su nombre dentro de esas 4 celdas y devuelve "Grupo 1/2/3/4" (o "SIN GRUPO"
  si no aparece en ninguna). La fórmula se recalcula sola cada vez que Fernando edite AD2/AE2/AF2/AG2.
- **INS NO CONSIDERADOS**: del catálogo de 21 instructores que dio Fernando, los que este mes NO
  tienen ningún vuelo en el CUADRO FINAL (no van a dictar LCK este mes) — lista estática, no
  fórmula, porque depende de los vuelos ya calculados arriba, no de lo que Fernando escriba en
  AD2:AG2.

In [ ]:
CATALOGO_INSTRUCTORES = [
    ("2713993", "Fiorella Ruiz"),
    ("2396710", "Sebastian Correa"),
    ("1271571", "Christian Rondon"),
    ("2843319", "Karen Santa Cruz"),
    ("3779550", "Jennifert Acurio"),
    ("29530", "Jazmin Guerra"),
    ("3217561", "Claudia Flores"),
    ("71348", "Karla Moz"),
    ("967092", "Erika Davila"),
    ("2369641", "Elizabeth Torres"),
    ("2369624", "Patricia Najar"),
    ("3134911", "Javier Zapata"),
    ("3750335", "Jefferson Mendez"),
    ("3852423", "Gabriela Ungaro"),
    ("2604360", "Mariella Carrasco"),
    ("2823133", "Cesar Campos"),
    ("3189967", "Kevin Segovia"),
    ("2415373", "Milagros Salas"),
    ("2440915", "Judith Fernandez"),
    ("3796947", "Rafael Nieto"),
    ("3841387", "Gianfranco Celiz"),
]  # lista dada directamente por Fernando (21 instructores), no derivada de ningún archivo

conteo_vuelos_por_ins = collections.Counter(f["INS"] for f in filas_cuadro_final)
instructores_con_vuelos = sorted(conteo_vuelos_por_ins.keys())
instructores_no_considerados = [nombre for _bp, nombre in CATALOGO_INSTRUCTORES
                                 if nombre not in conteo_vuelos_por_ins]

print(f"Instructores con vuelos en el CUADRO FINAL: {len(instructores_con_vuelos)}")
for nombre in instructores_con_vuelos:
    print(f"  {nombre}: {conteo_vuelos_por_ins[nombre]} vuelos")

print(f"\nInstructores del catálogo SIN ningún vuelo este mes (INS NO CONSIDERADOS): {len(instructores_no_considerados)}")
for nombre in instructores_no_considerados:
    print(f"  {nombre}")

# --- ubicaciones en Archivo 10 ---
FILA_HEADER_AUX = 14   # INS | CANTIDAD | GRUPOS
FILA_INICIO_AUX = 15   # confirmado: igual al AK15 del manual original
FILA_TOTAL_AUX = FILA_INICIO_AUX + len(instructores_con_vuelos)
COL_NO_CONSIDERADOS = "AO"


def formula_grupo(fila_ins):
    ref = f"$AK${fila_ins}"
    return (
        f'=SI(ESNUMERO(HALLAR({ref}; $AD$2)); "Grupo 1"; '
        f'SI(ESNUMERO(HALLAR({ref}; $AE$2)); "Grupo 2"; '
        f'SI(ESNUMERO(HALLAR({ref}; $AF$2)); "Grupo 3"; '
        f'SI(ESNUMERO(HALLAR({ref}; $AG$2)); "Grupo 4"; "SIN GRUPO"))))'
        
    )


filas_tabla_aux = []
for i, nombre in enumerate(instructores_con_vuelos):
    fila_ins = FILA_INICIO_AUX + i
    filas_tabla_aux.append([nombre, conteo_vuelos_por_ins[nombre], formula_grupo(fila_ins)])

print(f"\nTabla auxiliar: encabezado fila {FILA_HEADER_AUX} (AK:AM), datos desde fila {FILA_INICIO_AUX}, TOTAL en fila {FILA_TOTAL_AUX}")
for fila in filas_tabla_aux:
    print(fila)


## 10. Escribir tabla auxiliar + INS NO CONSIDERADOS en Archivo 10 — DESACTIVADO por defecto

In [ ]:
# --- DESCOMENTAR SOLO DESPUES DE REVISAR LA VISTA PREVIA ---

# ws_archivo10_lck320f.update(f"AK{FILA_HEADER_AUX}:AM{FILA_HEADER_AUX}", [["INS", "CANTIDAD", "GRUPOS"]])
# if filas_tabla_aux:
#     ws_archivo10_lck320f.update(
#         f"AK{FILA_INICIO_AUX}:AM{FILA_TOTAL_AUX - 1}",
#         filas_tabla_aux,
#         value_input_option="USER_ENTERED",  # necesario para que la columna GRUPOS quede como formula, no como texto
#     )
# ws_archivo10_lck320f.update(
#     f"AK{FILA_TOTAL_AUX}:AL{FILA_TOTAL_AUX}",
#     [["TOTAL", f"=SUM(AL{FILA_INICIO_AUX}:AL{FILA_TOTAL_AUX - 1})"]],
#     value_input_option="USER_ENTERED",
# )
# print(f"Tabla auxiliar escrita: {len(filas_tabla_aux)} instructores + fila TOTAL.")

# ws_archivo10_lck320f.update(f"{COL_NO_CONSIDERADOS}{FILA_HEADER_AUX}", [["INS NO CONSIDERADOS (solo nombres)"]])
# if instructores_no_considerados:
#     valores_no_considerados = [[n] for n in instructores_no_considerados]
#     ws_archivo10_lck320f.update(
#         f"{COL_NO_CONSIDERADOS}{FILA_HEADER_AUX + 1}:{COL_NO_CONSIDERADOS}{FILA_HEADER_AUX + len(valores_no_considerados)}",
#         valores_no_considerados,
#     )
#     print(f"'INS NO CONSIDERADOS' escrito: {len(valores_no_considerados)} nombres.")


## 11. QA: recalcular en Python que todo quede consistente

In [ ]:
# cada bloque asignado aparece como maximo 1 vez en el CUADRO FINAL (por pairing, 2 filas por bloque completo)
pares_esperados = sum(1 for _, (n, a) in asignaciones_por_bloque.items() if a in ACTIVIDADES_CUADRO_FINAL) * 2
print("Filas esperadas (2 por bloque con actividad LCK):", pares_esperados)
print("Filas realmente construidas:", len(filas_cuadro_final))
assert pares_esperados == len(filas_cuadro_final)

# ninguna fila del CUADRO FINAL con Cupos fuera de {3,4}
cupos_malos = [f for f in filas_cuadro_final if f["Cupos"] not in (3, 4)]
print("Filas con Cupos fuera de {3,4} (deben ser 0):", len(cupos_malos))

# columna Grupo del CUADRO FINAL debe quedar SIEMPRE vacia (confirmado con Fernando)
grupo_no_vacio = [f for f in filas_cuadro_final if f["Grupo"] != ""]
print("Filas con Grupo NO vacío en el CUADRO FINAL (deben ser 0):", len(grupo_no_vacio))

# cada instructor del catalogo esta en EXACTAMENTE una de las dos listas (con vuelos / no considerado)
nombres_catalogo = {nombre for _bp, nombre in CATALOGO_INSTRUCTORES}
cubiertos = set(instructores_con_vuelos) | set(instructores_no_considerados)
print("Catálogo completamente cubierto (con vuelos + no considerados):", nombres_catalogo == cubiertos)
print("Instructores con vuelos que no estan en el catalogo dado (revisar a mano):",
      set(instructores_con_vuelos) - nombres_catalogo)

# la cantidad de celdas de Matriz a actualizar coincide con la cantidad de bloques asignados
print("Celdas de Matriz a actualizar == bloques asignados:",
      len(actualizaciones_matriz) == len(asignaciones_por_bloque))


## 13. Por tripulante: cruzar "INS FINAL" (Y) contra los 4 grupos -> "INS F a considerar" (Z) y "Grupo" (AA)

La columna Y ("INS FINAL", instructores bloqueados por tripulante) **ya existe y no se toca**
(tiene su propia fórmula) — solo se lee. Lo que se calcula es Z y AA, para cada tripulante:

- Un grupo queda ELIMINADO si CUALQUIERA de sus integrantes aparece mencionado (nombre completo
  o apodo reconocido) en el texto de Y de ese tripulante — ese instructor (y por lo tanto su
  grupo completo) no debe ser considerado para chequear a ese tripulante.
- **AA ("Grupo")**: los números de los grupos que SÍ quedan disponibles, formato `"Grupo 1/2/3"`.
- **Z ("INS F a considerar")**: los NOMBRES de los integrantes de cada grupo disponible — dentro
  de un grupo separados por coma, entre grupos separados por `" / "` (formato confirmado por
  Fernando: `"Patricia, Cristian, Fiore / Juan, Felipe"`).

**El texto de la columna Y es muy inconsistente** (nombre completo, solo primer nombre, apodos
como "Sebas"/"Fio"/"Cris", separadores "/" o "y" o ",", "#N/A", resúmenes libres). Por eso la
detección de qué grupo bloquea cada fila se hace con Python + una tabla de apodos conocidos
(sacada de los datos reales que pasó Fernando), no con una fórmula de Sheets.

In [ ]:
# Apodos/variantes reales encontrados en los datos que pasó Fernando (columna Y real).
# Solo hacen falta para los integrantes de los 4 grupos -> bloqueos a OTROS instructores
# (Milagros Salas, Kevin Segovia, Erika Davila, Gianfranco Celiz, Cesar Campos, Judith
# Fernandez, etc.) no afectan el cálculo porque no pertenecen a ningún grupo.
ALIAS_INSTRUCTORES = {
    "Fiorella Ruiz": ["Fiorella", "Fio", "Fiore"],
    "Sebastian Correa": ["Sebastian", "Sebastián", "Sebas"],
    "Christian Rondon": ["Christian", "Cristian", "Cris"],
    "Claudia Flores": ["Claudia"],
    "Elizabeth Torres": ["Elizabeth"],
    "Gabriela Ungaro": ["Gabriela"],
    "Javier Zapata": ["Javier"],
    "Karen Santa Cruz": ["Karen"],
    "Jazmin Guerra": ["Jazmin", "Jazmín"],
    "Jefferson Mendez": ["Jefferson"],
    "Jennifert Acurio": ["Jennifert"],
    "Mariella Carrasco": ["Mariella"],
    "Patricia Najar": ["Patricia"],
}


def normalizar_texto_libre(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return s.lower()


def nombre_mencionado(nombre_canonico, texto):
    texto_norm = normalizar_texto_libre(texto)
    candidatos = [nombre_canonico] + ALIAS_INSTRUCTORES.get(nombre_canonico, [])
    for candidato in candidatos:
        patron = r"\b" + re.escape(normalizar_texto_libre(candidato)) + r"\b"
        if re.search(patron, texto_norm):
            return True
    return False


# Grupos leidos EN VIVO de la hoja real (las mismas 4 celdas que llena Fernando a mano) ->
# no se hardcodea la composicion de los grupos, porque ya cambio una vez desde que se
# documento originalmente.
texto_grupo1 = ws_archivo10_lck320f.acell("AD2").value or ""
texto_grupo2 = ws_archivo10_lck320f.acell("AE2").value or ""
texto_grupo3 = ws_archivo10_lck320f.acell("AF2").value or ""
texto_grupo4 = ws_archivo10_lck320f.acell("AG2").value or ""

grupos_definidos = {
    1: [n.strip() for n in texto_grupo1.split(",") if n.strip()],
    2: [n.strip() for n in texto_grupo2.split(",") if n.strip()],
    3: [n.strip() for n in texto_grupo3.split(",") if n.strip()],
    4: [n.strip() for n in texto_grupo4.split(",") if n.strip()],
}
print("Grupos leídos en vivo de AD2/AE2/AF2/AG2:")
for num, nombres in grupos_definidos.items():
    print(f"  Grupo {num}: {nombres}")


In [ ]:
valores_hoja = ws_archivo10_lck320f.get_all_values()

COL_BP = 0  # columna A
fila_header_roster = None
COL_INS_FINAL = None
for i, fila in enumerate(valores_hoja):
    fila_norm = [c.strip().upper() for c in fila]
    if "INS FINAL" in fila_norm:
        fila_header_roster = i
        COL_INS_FINAL = fila_norm.index("INS FINAL")
        break

if fila_header_roster is None:
    raise RuntimeError("No encontré el encabezado 'INS FINAL' en Archivo 10 -> revisar estructura real de la hoja.")

print(f"Encabezado 'INS FINAL' encontrado en fila {fila_header_roster + 1} (1-indexado), columna {COL_INS_FINAL + 1}")

filas_tripulantes = []  # (fila_hoja, bp, texto_ins_final)
for i, fila in enumerate(valores_hoja[fila_header_roster + 1:], start=fila_header_roster + 2):
    if not fila or not fila[COL_BP].strip():
        continue
    bp = fila[COL_BP].strip()
    texto_y = fila[COL_INS_FINAL] if COL_INS_FINAL < len(fila) else ""
    filas_tripulantes.append((i, bp, texto_y))

print(f"Filas de tripulantes leídas (con BP no vacío): {len(filas_tripulantes)}")


In [ ]:
def texto_indica_sin_bloqueo(texto):
    t = texto.strip().upper()
    return t in ("", "-", "#N/A", "N/A")


resultados_grupo = []  # (fila_hoja, bp, texto_y, considerados_texto, grupo_texto)

for fila_hoja, bp, texto_y in filas_tripulantes:
    if texto_indica_sin_bloqueo(texto_y):
        grupos_elegibles = [1, 2, 3, 4]
    else:
        grupos_elegibles = []
        for num, nombres_grupo in grupos_definidos.items():
            bloqueado = any(nombre_mencionado(nombre, texto_y) for nombre in nombres_grupo)
            if not bloqueado:
                grupos_elegibles.append(num)

    grupo_texto = ("Grupo " + "/".join(str(n) for n in grupos_elegibles)) if grupos_elegibles else "SIN GRUPO"
    # Z ("INS F a considerar"): nombres de los integrantes de cada grupo ELEGIBLE (no los
    # bloqueados) -> dentro de un grupo separados por coma, entre grupos separados por " / "
    # (formato confirmado por Fernando: "Patricia, Cristian, Fiore / Juan, Felipe").
    considerados_texto = " / ".join(", ".join(grupos_definidos[n]) for n in grupos_elegibles)
    resultados_grupo.append((fila_hoja, bp, texto_y, considerados_texto, grupo_texto))

print(f"Tripulantes procesados: {len(resultados_grupo)}\n")
for fila_hoja, bp, texto_y, considerados_texto, grupo_texto in resultados_grupo:
    print(f"fila {fila_hoja} | BP {bp} | Y='{texto_y}' -> Z='{considerados_texto}' | AA='{grupo_texto}'")

conteo_resultado = collections.Counter(g for *_r, g in resultados_grupo)
print("\nDistribución de resultados de Grupo:")
for grupo_texto, n in conteo_resultado.most_common():
    print(f"  {grupo_texto}: {n} tripulantes")


## 14. Escribir "INS F a considerar" (Z) y "Grupo" (AA) por tripulante — DESACTIVADO por defecto

In [ ]:
# --- DESCOMENTAR SOLO DESPUES DE REVISAR LA VISTA PREVIA ---

# # COL_INS_FINAL es 0-indexado (posición de "Y" en la lista de Python) -> convertir a
# # columna 1-indexada de gspread y sumar el desplazamiento a Z (+1) y AA (+2).
# col_z_1idx = COL_INS_FINAL + 2   # Y(0-idx) + 1 = Z(0-idx); +1 para volverlo 1-indexado
# col_aa_1idx = COL_INS_FINAL + 3  # Y(0-idx) + 2 = AA(0-idx); +1 para volverlo 1-indexado
# celdas_zaa = []
# for fila_hoja, bp, texto_y, considerados_texto, grupo_texto in resultados_grupo:
#     celdas_zaa.append(gspread.Cell(row=fila_hoja, col=col_z_1idx, value=considerados_texto))
#     celdas_zaa.append(gspread.Cell(row=fila_hoja, col=col_aa_1idx, value=grupo_texto))
# if celdas_zaa:
#     ws_archivo10_lck320f.update_cells(celdas_zaa, value_input_option="USER_ENTERED")
#     print(f"Escritas {len(celdas_zaa)} celdas (Z + AA) para {len(resultados_grupo)} tripulantes.")


## 15. QA: recalcular en Python el cruce Y -> Z/AA

In [ ]:
# todo resultado de Grupo debe ser "SIN GRUPO" o empezar con "Grupo "
formatos_malos = [g for *_r, g in resultados_grupo if g != "SIN GRUPO" and not g.startswith("Grupo ")]
print("Resultados de Grupo con formato inesperado (deben ser 0):", len(formatos_malos))

# si Y indica "sin bloqueo" (vacío/-/#N/A), el resultado debe ser el grupo completo
sin_bloqueo_pero_no_completo = [
    (bp, texto_y, g) for _f, bp, texto_y, _c, g in resultados_grupo
    if texto_indica_sin_bloqueo(texto_y) and g != "Grupo 1/2/3/4"
]
print("Tripulantes sin bloqueo (Y vacío/-/N-A) que NO quedaron con los 4 grupos (deben ser 0):",
      len(sin_bloqueo_pero_no_completo))

# cuantos tripulantes quedaron sin ningun grupo disponible -> revisar a mano, caso raro
sin_ningun_grupo = [(bp, texto_y) for _f, bp, texto_y, _c, g in resultados_grupo if g == "SIN GRUPO"]
print(f"Tripulantes SIN NINGÚN grupo disponible (revisar a mano): {len(sin_ningun_grupo)}")
for bp, texto_y in sin_ningun_grupo:
    print(f"  BP {bp}: Y='{texto_y}'")


## 17. Fase 5 (Freeze): armar el roster + tabla CUADRO FINAL alterna

Reutiliza todo lo ya calculado (`grupos_definidos`, `resultados_grupo`, `filas_cuadro_final`,
`valores_hoja` de Archivo 10) — no se vuelve a consultar BigQuery ni a leer la Matriz.

**Layout confirmado por Fernando** (hoja de ejemplo del Freeze, mismo spreadsheet que la Matriz,
gid `1346457970`) — **solo 2 tablas**, sin bloque de vuelos aparte:
- **Roster** desde `A1` (encabezado) / `A2` (datos): `BP · CAT · Nombre · Estado · Vigencia ·
  Comentario · Grupo`, solo tripulantes con `PROGRAMAR = Sí` en Archivo 10.
- **Tabla alterna** desde `L5`: encabezado "CUADRO FINAL ( LCK , DGAC)" en L5, columnas en L6
  (Fecha/DíaSEM/Vuelo/Ruta/N° Cupos/Grupo/INS), datos desde L7 — mismo layout que el CUADRO
  FINAL de Archivo 10 (label/encabezado/datos), pero acá con el Grupo del INSTRUCTOR sí
  completado (a qué grupo pertenece cada instructor asignado).

**De dónde sale cada dato (confirmado por Fernando, columnas de Archivo 10):**
- BP = columna A, CAT = columna B, Nombre = columna C, Grupo = columna AA (ya calculado en
  `resultados_grupo`, sección 13).
- Vigencia y PROGRAMAR/OBS FREEZE se buscan por texto de encabezado (como con "INS FINAL"), no
  se hardcodea la letra de columna, para no depender de una lectura de voz ambigua.
- **Comentario** = `{OBS FREEZE}. Programar como parte de la tripulación.` (concatenado,
  confirmado por Fernando).
- **Estado**: no se especificó de dónde sale: en el ejemplo real todas las filas mostraban "-",
  así que se deja como `"-"` por defecto — avisar si debe salir de otro lado.


In [ ]:
def buscar_columna(fila_header, texto_buscado):
    fila_norm = [c.strip().upper() for c in fila_header]
    texto_norm = texto_buscado.strip().upper()
    if texto_norm not in fila_norm:
        raise RuntimeError(f"No encontré la columna '{texto_buscado}' en el encabezado de Archivo 10 -> revisar estructura real de la hoja.")
    return fila_norm.index(texto_norm)


fila_header = valores_hoja[fila_header_roster]
COL_CAT = buscar_columna(fila_header, "CAT")
COL_NOMBRE = buscar_columna(fila_header, "Nombre")
COL_VIGENCIA = buscar_columna(fila_header, "Vigencia")
COL_PROGRAMAR = buscar_columna(fila_header, "PROGRAMAR")
COL_OBS_FREEZE = buscar_columna(fila_header, "OBS FREEZE")

print(f"CAT: columna {COL_CAT + 1} | Nombre: columna {COL_NOMBRE + 1} | Vigencia: columna {COL_VIGENCIA + 1}")
print(f"PROGRAMAR: columna {COL_PROGRAMAR + 1} | OBS FREEZE: columna {COL_OBS_FREEZE + 1}")


def es_si(texto):
    return texto.strip().lower() in ("si", "sí")


datos_freeze = []  # (fila_hoja, bp, cat, nombre, vigencia, programar, obs_freeze)
for i, fila in enumerate(valores_hoja[fila_header_roster + 1:], start=fila_header_roster + 2):
    if not fila or not fila[COL_BP].strip():
        continue
    bp = fila[COL_BP].strip()
    cat = fila[COL_CAT].strip() if COL_CAT < len(fila) else ""
    nombre = fila[COL_NOMBRE].strip() if COL_NOMBRE < len(fila) else ""
    vigencia = fila[COL_VIGENCIA].strip() if COL_VIGENCIA < len(fila) else ""
    programar = fila[COL_PROGRAMAR].strip() if COL_PROGRAMAR < len(fila) else ""
    obs_freeze = fila[COL_OBS_FREEZE].strip() if COL_OBS_FREEZE < len(fila) else ""
    datos_freeze.append((i, bp, cat, nombre, vigencia, programar, obs_freeze))

print(f"\nFilas de tripulantes leídas: {len(datos_freeze)}")


In [ ]:
grupo_por_fila = {fila_hoja: grupo_texto for fila_hoja, _bp, _y, _z, grupo_texto in resultados_grupo}

filas_roster_freeze = []  # dicts: BP, CAT, Nombre, Estado, Vigencia, Comentario, Grupo
for fila_hoja, bp, cat, nombre, vigencia, programar, obs_freeze in datos_freeze:
    if not es_si(programar):
        continue
    grupo_texto = grupo_por_fila.get(fila_hoja, "")
    obs_limpio = obs_freeze.rstrip(".").strip()
    comentario = (
        f"{obs_limpio}. Programar como parte de la tripulación."
        if obs_limpio else "Programar como parte de la tripulación."
    )
    filas_roster_freeze.append({
        "BP": bp, "CAT": cat, "Nombre": nombre, "Estado": "-",
        "Vigencia": vigencia, "Comentario": comentario, "Grupo": grupo_texto,
    })

print(f"Tripulantes con PROGRAMAR = Sí para el Freeze: {len(filas_roster_freeze)}")
print("\nPrimeras filas del roster:")
for f in filas_roster_freeze[:6]:
    print(f)


## 18. Escribir el roster + bloque de vuelos en la hoja de ejemplo del Freeze — DESACTIVADO por defecto

In [ ]:
# Misma spreadsheet que la Matriz (sh_matriz), solo cambia la pestaña (gid).
ws_freeze = sh_matriz.get_worksheet_by_id(1346457970)
print("Hoja de ejemplo del Freeze abierta:", ws_freeze.title)

encabezado_roster = [["BP", "CAT", "Nombre", "Estado", "Vigencia", "Comentario", "Grupo"]]
valores_roster = [
    [f["BP"], f["CAT"], f["Nombre"], f["Estado"], f["Vigencia"], f["Comentario"], f["Grupo"]]
    for f in filas_roster_freeze
]

print(f"Se escribirían {len(valores_roster)} filas de roster (A2:G{1 + len(valores_roster)})")

# --- DESCOMENTAR SOLO DESPUES DE REVISAR LA VISTA PREVIA ---
# ws_freeze.update("A1", encabezado_roster)
# if valores_roster:
#     ws_freeze.update(f"A2:G{1 + len(valores_roster)}", valores_roster)
# print("Roster escrito en la hoja de ejemplo del Freeze.")


## 19. Vista previa: tabla alterna en L5 (CUADRO FINAL con Grupo del instructor)

A diferencia del CUADRO FINAL de Archivo 10 (que deja Grupo vacío), acá sí se completa: es el
grupo AL QUE PERTENECE el instructor asignado (búsqueda directa en `grupos_definidos`, no
necesita la lógica de Y/apodos). Si algún instructor asignado no está en ninguno de los 4
grupos, se marca "SIN GRUPO" en vez de inventarlo — no debería pasar si viene de un bloque ya
asignado, pero se reporta por las dudas.

In [ ]:
instructor_a_grupo_num = {}
for num, nombres in grupos_definidos.items():
    for nombre in nombres:
        instructor_a_grupo_num[nombre] = num

filas_cuadro_final_freeze = []  # dicts: Fecha, DiaSem, Vuelo, Ruta, Cupos, Grupo, INS
instructores_sin_grupo_cuadro = set()

for f in filas_cuadro_final:
    num = instructor_a_grupo_num.get(f["INS"])
    if num is None:
        instructores_sin_grupo_cuadro.add(f["INS"])
        grupo_val = "SIN GRUPO"
    else:
        grupo_val = f"Grupo {num}"
    filas_cuadro_final_freeze.append({
        "Fecha": f["Fecha"], "DiaSem": f["DiaSem"], "Vuelo": f["Vuelo"], "Ruta": f["Ruta"],
        "Cupos": f["Cupos"], "Grupo": grupo_val, "INS": f["INS"],
    })

if instructores_sin_grupo_cuadro:
    print("AVISO: instructores en el CUADRO FINAL sin grupo fijo definido (revisar a mano):",
          instructores_sin_grupo_cuadro)

print(f"Filas para la tabla alterna (L5): {len(filas_cuadro_final_freeze)}")
print("\nPrimeras filas:")
for f in filas_cuadro_final_freeze[:6]:
    print(f)


## 20. Escribir la tabla alterna en L5 — DESACTIVADO por defecto

In [ ]:
FILA_LABEL_L5 = 5
FILA_HEADER_L5 = 6
FILA_DATOS_L5 = 7
COL_INICIO_L5 = "L"

valores_l5 = [
    [f["Fecha"], f["DiaSem"], f["Vuelo"], f["Ruta"], f["Cupos"], f["Grupo"], f["INS"]]
    for f in filas_cuadro_final_freeze
]
print(f"Se escribiría la tabla alterna en L5 con {len(valores_l5)} filas de datos (desde L{FILA_DATOS_L5})")

# --- DESCOMENTAR SOLO DESPUES DE REVISAR LA VISTA PREVIA ---
# ws_freeze.update(f"{COL_INICIO_L5}{FILA_LABEL_L5}", [["CUADRO FINAL ( LCK , DGAC)"]])
# ws_freeze.update(
#     f"{COL_INICIO_L5}{FILA_HEADER_L5}:R{FILA_HEADER_L5}",
#     [["Fecha", "DíaSEM", "Vuelo", "Ruta", "N° Cupos", "Grupo", "INS"]],
# )
# if valores_l5:
#     ws_freeze.update(f"{COL_INICIO_L5}{FILA_DATOS_L5}:R{FILA_DATOS_L5 + len(valores_l5) - 1}", valores_l5)
# print(f"Tabla alterna escrita en L5: {len(valores_l5)} filas.")


## 21. QA: recalcular en Python la Fase 5

In [ ]:
# cada tripulante del roster de Freeze debe tener PROGRAMAR = Sí (por construcción, se verifica igual)
bps_roster = {f["BP"] for f in filas_roster_freeze}
bps_programar_si = {bp for _f, bp, _c, _n, _v, programar, _o in datos_freeze if es_si(programar)}
print("BPs del roster == BPs con PROGRAMAR=Sí:", bps_roster == bps_programar_si)

# ningun BP duplicado en el roster del Freeze
conteo_bp = collections.Counter(f["BP"] for f in filas_roster_freeze)
duplicados = [bp for bp, n in conteo_bp.items() if n > 1]
print("BPs duplicados en el roster (deben ser 0):", len(duplicados))

# la tabla alterna L5 tiene la misma cantidad de filas que el CUADRO FINAL, y ningun Grupo vacio
print("Filas de la tabla alterna == filas del CUADRO FINAL:",
      len(filas_cuadro_final_freeze) == len(filas_cuadro_final))
grupo_vacio_l5 = [f for f in filas_cuadro_final_freeze if not f["Grupo"]]
print("Filas de la tabla alterna con Grupo vacío (deben ser 0):", len(grupo_vacio_l5))


## 22. Estado y pendientes — sin inventar nada

### Lo que este notebook automatiza
- Reemplaza el slot `"LCK A320F"` de la Matriz por el detalle real de vuelos (formato confirmado
  por Fernando), en modo vista previa.
- Arma el CUADRO FINAL para Archivo 10 (Fecha, DíaSEM, Vuelo, Ruta, Cupos, INS), filtrando por
  actividad LCK, con la columna Grupo **vacía** (confirmado por Fernando).
- Arma la tabla auxiliar (INS / Cantidad / Grupos, en `AK14:AM...`) con la misma **fórmula** que
  Fernando ya usa en la hoja real, leyendo las 4 celdas donde escribe a mano los grupos
  (`AD2`/`AE2`/`AF2`/`AG2`) — el notebook no decide la composición de los grupos.
- Arma la lista "INS NO CONSIDERADOS" (instructores del catálogo sin ningún vuelo este mes).
- **Por tripulante**, cruza la columna Y ("INS FINAL", ya existente y sin tocar) con los 4 grupos
  leídos en vivo, y calcula "INS F a considerar" (Z, formato `"Patricia, Cristian, Fiore / Juan,
  Felipe"`) y "Grupo" (AA, formato `"Grupo 1/2/3"`).
- **Fase 5 (Freeze)**: arma el roster de tripulantes con `PROGRAMAR = Sí` (BP/CAT/Nombre/Estado/
  Vigencia/Comentario/Grupo, desde A1) y la tabla alterna del CUADRO FINAL con el Grupo del
  instructor completado (desde L5) — **solo esas 2 tablas**, sin bloque de vuelos aparte
  (confirmado por Fernando: eso pisaba la tabla alterna).

### Supuestos hechos (revisar antes de confiar en el resultado real)
- **"#N/A", "-" y celdas vacías en Y** se interpretan como "sin bloqueo" -> el tripulante queda
  con los 4 grupos disponibles.
- **Tabla de apodos** (Sebas, Fio, Fiore, Cris, etc.): cubre las variantes vistas en los datos
  reales que pasó Fernando para los 13 integrantes de los 4 grupos. Un apodo nuevo no reconocido
  no se va a detectar — por eso la vista previa imprime el texto original de Y junto al
  resultado.
- **Comentario del Freeze**: se asumió `"{OBS FREEZE}. Programar como parte de la tripulación."`
  — confirmado por Fernando, pero la puntuación exacta (punto vs. dos puntos) es una
  interpretación de lo dictado; revisar el resultado real antes de dar por bueno el formato.
- **"Estado" del Freeze**: no se especificó su origen; se deja `"-"` por defecto (así aparecía en
  el ejemplo real que se vio). Avisar si debe salir de otro campo.
- **Roster del Freeze filtrado por `PROGRAMAR = Sí`**: es la misma señal de demanda usada en toda
  la Fase 1 — no se confirmó explícitamente para este paso puntual, pero es consistente con "la
  población que requería LCK" que menciona el documento (~206 tripulantes).

### Lo que sigue sin automatizar
- **Los propios grupos de instructores** (AD2/AE2/AF2/AG2) — Fernando los escribe a mano cada mes.
- **La columna Y ("INS FINAL")** — ya viene resuelta desde otro proceso, no se reconstruye acá.
- El destino "Rol Instructores... para asignar los vuelos a cada instructor" que menciona el
  documento, más allá de lo que ya hace este notebook — no se automatizó nada adicional ahí.
- Validación de capacidad por grupo (paso 21 del documento original) y conciliación cupos vs.
  demanda (pasos 23-25 de Fase 5: TJ por vuelo, priorización por vencimiento, envío a Karina,
  handoff a Diana por cancelaciones) — siguen siendo revisión/decisión manual, no hay reglas lo
  suficientemente formales en el documento para automatizarlas sin inventar criterio.
